# RAG: Haverford College Concert Programs Fall 2009 - Spring 2022

## Goals

* Create a Retrieval Augmented Generation app for Haverford Concert programs 2009-2022
* Implement metadata filters for more refined context, thus higher chance of accurate responses

If the kernel restarts, do not run all!! Instead, run the code [here](full_concert_programs.ipynb#run-this-cell-to-reestablish-variables-and-keys).

## Imports and Initial Setup

In [1]:
from typing_extensions import List, TypedDict
import getpass
import os
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

In [3]:
# Setting up chat model

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")


llm = init_chat_model("gpt-4o-mini", model_provider="openai")
# Setting up embeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
# Setting up chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

## Setting Up Retrieval and Generation

We use LangGraph and a State class to set up our RAG. LangGraph allows us to set up clear steps, and the State allows us to pass information between each step. 

See the graph below the 3 code cells.

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use only the information provided in the context below to answer the question. If the answer is not in the context, say 'I don't know' or 'The information is not available.'"),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"], k = 10)
    return {"context": retrieved_docs}

def generate(state: State):
    docs_content = "\n\n".join([doc.page_content for doc in state["context"]])
    message = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(message)
    return {"answer": response.content}


In [5]:
from langgraph.graph import START, StateGraph

graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

In [6]:
print(graph.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+----------+   
| retrieve |   
+----------+   
      *        
      *        
      *        
+----------+   
| generate |   
+----------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


## Run this cell to reestablish variables and keys

In [7]:
# Setting up chat model
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

# Setting up embeddings
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# Reestablishing up our persist directory for Chroma
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where we stored our data before
)

# Reestablishing retrieval and generation functions
from langchain_core.documents import Document
from typing_extensions import List, TypedDict

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use only the information provided in the context below to answer the question. If the answer is not in the context, say 'I don't know' or 'The information is not available.'"),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"], k = 10)
    return {"context": retrieved_docs}

def generate(state: State):
    docs_content = "\n\n".join([doc.page_content for doc in state["context"]])
    message = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(message)
    return {"answer": response.content}

# Reestablishing langgraph
from langgraph.graph import START, StateGraph

graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

## The Basic RAG is set up - Let's ask some questions about our data

In [26]:
result = graph.invoke({"question": "Who has conducted the Chamber Singers over the years?"})

print(f'Answer: {result["answer"]}\n')
print(f'Context: {result["context"]}\n\n')



Answer: The Chamber Singers have been conducted by Professor Thomas Lloyd, Ng Tian Hui, and Dr. Nathan Zullinger.

Context: [Document(id='13720808-826c-45c8-b695-85c2b85d9445', metadata={'start_index': 22880, 'CreationDate': "D:20160524182813Z00'00'", 'total_pages': 12, 'Term': 'Spring', 'page': 0, 'file_path': 'C:\\Users\\charl\\Documents\\VSCode\\HC All Programs\\Files\\PDFs\\30. Chamber Singers-Recalling Ariadne Program.pdf', 'source': '30. Chamber Singers-Recalling Ariadne Program.pdf', 'Ensemble_Type': 'Choral', 'Year': '2011'}, page_content='academic year of 2010/11, the choir is led by Ng Tian Hui while Prof. Lloyd is away on sabbatical.\nAs the premier vocal ensemble in the bi-college community, the Chamber Singers performs challenging repertoire ranging from\nthe Renaissance to the present day in a variety of languages and styles. This season’s repertoire will range from 16th century\nParis to contemporary South Africa; from Bach’s Magnificat, to Monteverdi’s Lamento d’Arianna

In [30]:
# better formatting, with just the context for each response element
print(f'Answer: {result["answer"]}\n')

# context is a list of Document objects
for i, doc in enumerate(result["context"]):
    print(f'Context {i+1}: {doc.metadata.get("Year", "unknown")}\n{doc.page_content}\n')


Answer: The Chamber Singers have been conducted by Professor Thomas Lloyd, Ng Tian Hui, and Dr. Nathan Zullinger.

Context 1: 2011
academic year of 2010/11, the choir is led by Ng Tian Hui while Prof. Lloyd is away on sabbatical.
As the premier vocal ensemble in the bi-college community, the Chamber Singers performs challenging repertoire ranging from
the Renaissance to the present day in a variety of languages and styles. This season’s repertoire will range from 16th century
Paris to contemporary South Africa; from Bach’s Magnificat, to Monteverdi’s Lamento d’Arianna, to new commissions by
composers based in Austria, Singapore and the US.
The Chamber Singers returned from Turkey and their sixth international cultural exchange tour in the spring of 2010, and will
embark on a national tour in Spring 2011 including an exchange with Amherst College and a performance at Yale University’s
Marquand Chapel. In addition, the choir looks forward to a session with Meredith Monk, and an invitatio

In [31]:
result = graph.invoke({"question": "Who played trumpet for the haverford-bryn mawr orchestra in 2019?"})
# context is a list of Document objects
for i, doc in enumerate(result["context"]):
    print(f'Context {i+1}: {doc.metadata.get("Year", "unknown")}\n{doc.page_content}\n')

Context 1: 2011
Haverford/Bryn Mawr Chorale Orchestra – Fall 2011
Violin I Bass Contrabassoon
Lorenzo Rayal Jennifer Bradbury Ben Hoyle
Kiran Rajamani HC ’14 Matt Roberts
Chi Park Brent Edmondson Horns
Natalia Banfi HC ’15 Katie Jordan
Harp
Jennifer Horne Kristina Gannon
Ariane Giles HC ’15
Vanessa Felso BMC ’15 Sabrina Huber
Sophia Forker HC ’15
Vena Johnson Ryan Stewart
Gabriella Goodman HC ’12
Piccolo
Trumpets
Violin II Katherine Barbato
Brian Rascon
Rodolfo Leuenberger
Matthew Thomas
Marie Greaney HC ’14 Flute
Aaron Matthias-Long
Angela Sulzer BJ Hillinck HC ’15
Nora Schmidt BMC ’12 Evangeline Krajewski HC ’14
Trombones
Sarah Franklin
Brian Santero
Lynn Richmond HC ’13 Oboe
Keith Warner HC ’15
Megan Reimer
Anthony Triplett
Viola
Dario McConnie-Saad English Horn
Tuba
Esther Ho BMC ’15 Fiona Last
Chris Pearlberg
Sarah Pisano
Kristina Kronauer BMC ’13 Clarinet
Percussion
Alyssa DeStefano Tom Lee HC ’15
Tim Knowlten
Veeshal Modi Leslie Tjing HC ’15
Seth Bagwell
Cello Brad Broomfield
Ba

Clearly, we didn't pull the right documents in the above prompt. This is where the metadata filters will come in, once we set them up.

In [32]:
result = graph.invoke({"question": "Who are some trumpet players in the haverford-bryn mawr orchestra?"})

print(f'Answer: {result["answer"]}')
print("\nSources:")
for source in result["context"]:
    print(f'Source: {source.metadata["source"]}')

Answer: Some trumpet players in the Haverford-Bryn Mawr Orchestra include Christian Fagre, Sam Istvan, and Andrew Cornell.

Sources:
Source: 19. Chorale Program Spring 2013.pdf
Source: 9. Chorale Program Fall 2011.pdf
Source: 6. Chorale Program Fall 2012.pdf
Source: HC Family Weekend 08 program.pdf
Source: 8. Chorale Program Fall 2019.pdf
Source: 7. Orchestra Program Fall 2018.pdf
Source: 2. Family Weekend Choral Program-2013.pdf
Source: 10. Chorale Program Fall 2010.pdf
Source: 8. Chorale Program Fall 2018.pdf
Source: HC Family Weekend 08 program.pdf


## Setting up Filters

As you can see, it's working pretty well, but the similarity search isn't always pulling the right documents. If we could add a simple filter, such as specifying the year or ensemble type, our results would be much better, without having to do any extra calls to the API.

Setting up a filter in similarity search:

In [33]:
k = 4
ensemble_filter = {"Ensemble_Type": "Orchestra"}
vector_store.similarity_search("Who was the principle trumpet player in the Haverford-Bryn Mawr Orchestra in 2019?", k=k, filter=ensemble_filter)

[Document(id='6178eb27-1284-4041-8289-1d804847fcf7', metadata={'page': 0, 'file_path': 'C:\\Users\\charl\\Documents\\VSCode\\HC All Programs\\Files\\PDFs\\9. Orchestra Program Fall 2010.pdf', 'start_index': 776, 'Ensemble_Type': 'Orchestra', 'total_pages': 7, 'source': '9. Orchestra Program Fall 2010.pdf', 'Year': '2010', 'Term': 'Fall', 'CreationDate': "D:20160524173625Z00'00'"}, page_content='Josh Bucheister, ’14, Associate Rachael Goldstein, ’14 Co- Principal Katie Van Aken, ’12 %\nConcertmaster Ezekiel Barnett, ’13 Assistant Principal Elizabeth Biernat, ’14@\nEthan Joseph, ’11, Assistant Concertmaster Delaney Page, ’12 Kristina Kronauer, ’13\nXinbei Guan, ’14, Assistant Concertmaster Laura Alexander, ’11 Kristina Gannon **\nSarah Capasso, ’11 Noory O, ’13\nDrew Twitchell, ’11 Erin Korth, ’13 Trumpet\nNora Schmidt, ’12 Mary Schultz, ’12 Ian Gavigan, ’14 &@\nTiffany Fritz, ’12 Seoung Won Jung, ’14 Linus Marco, ’13\nYiran Zhang , ’14 Kathryn Hayden, ’14 Chelsea Miller, ’11* %\nMarie G

Here, we repeat content from earlier, making this our new "reestablishment" cell. Running this cell allows the RAG to work. Then, we add filters within the LangGraph.

In [12]:
# Setting up chat model
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

# Setting up embeddings
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# Reestablishing up our persist directory for Chroma
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where we stored our data before
)

# Reestablishing retrieval and generation functions
from langchain_core.documents import Document
from typing_extensions import List, TypedDict, Optional
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use only the information provided in the context below to answer the question. If the answer is not in the context, say 'I don't know' or 'The information is not available.'"),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

class State(TypedDict):
    question: str
    filter: Optional[dict]
    context: List[Document]
    answer: str

def apply_filter(state: State):
    """
    Apply a filter to the state based on the user's input.
    The filter is expected to be a dictionary that can be used in the similarity search.
    """
    if "filter" in state and state["filter"]:
        # If a filter is provided, use it
        state["filter"] = {k: v for k, v in state["filter"].items() if v is not None}
    else:
        # If no filter is provided, set it to None
        state["filter"] = None
    return state

def retrieve(state: State):
    filter_dict = state["filter"] if state.get("filter") else None
    retrieved_docs = vector_store.similarity_search(state["question"], k=10, filter=filter_dict)
    return {"context": retrieved_docs}

def generate(state: State):
    docs_content = "\n\n".join([doc.page_content for doc in state["context"]])
    message = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(message)
    return {"answer": response.content}

# Reestablishing langgraph
from langgraph.graph import START, StateGraph

graph_builder = StateGraph(State).add_sequence([apply_filter, retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

In [35]:
question = "Who played trumpet for the orchestra in the year 2019?"
ensemble_filter = {"Ensemble_Type": "Orchestra"}


result = graph.invoke({
    "question": question, "filter": ensemble_filter})

print(f'Answer: {result["answer"]}')
print("\n\nSources:")
for i, source in enumerate(result["context"]):
    print(f'Source {i+1}: {source.metadata["source"]}')

Answer: The information is not available.


Sources:
Source 1: 21. Orchestra Program-Spring 2012.pdf
Source 2: 17. Orchestra Program Spring 2013.pdf
Source 3: 9. Orchestra Program Fall 2010.pdf
Source 4: 19. Orchestra Program Spring 2016.pdf
Source 5: Fall 11-19-21 Orchestra Program Fall 2021.pdf
Source 6: 7. Orchestra Program Fall 2011.pdf
Source 7: 4. Orchestra Program Fall 2012.pdf
Source 8: 5. Orchestra Program Fall 2014.pdf
Source 9: 8. Orchestra Program Fall 2016.pdf
Source 10: 27. Orchestra Program-Spring 2011.pdf


In [36]:
result = graph.invoke({
    "question": "Who played trumpet for the haverford-bryn mawr orchestra in 2019?",
    "filter": {"$and": [{"Ensemble_Type": "Orchestra"}, {"Year": "2019"}]}
})

print(f'Answer: {result["answer"]}')
print("\n\nSources:")
for i, source in enumerate(result["context"]):
    print(f'Source {i+1}: {source.metadata["source"]}')

Answer: The trumpet players for the Haverford-Bryn Mawr College Orchestra in 2019 included Sam Istvan, who was the Principal, and Jackie Toben, who was the Co-Associate Principal.


Sources:
Source 1: 7. Orchestra Program Fall 2019.pdf
Source 2: 7. Orchestra Program Fall 2019.pdf
Source 3: 18. Orchestra Program Spring 2019.pdf
Source 4: 7. Orchestra Program Fall 2019.pdf
Source 5: 18. Orchestra Program Spring 2019.pdf
Source 6: 18. Orchestra Program Spring 2019.pdf
Source 7: 7. Orchestra Program Fall 2019.pdf
Source 8: 7. Orchestra Program Fall 2019.pdf
Source 9: 18. Orchestra Program Spring 2019.pdf
Source 10: 18. Orchestra Program Spring 2019.pdf


In [37]:
result = graph.invoke({
    "question": "what are the types of music performed at the orchestra concerts?", "filter" :{"Ensemble_Type": "Orchestra"}
})
#more specific questions, interesting that they answered the question. accuracy is uncertain. 
print(f'Answer: {result["answer"]}')
print("\n\nSources:")
for i, source in enumerate(result["context"]):
    print(f'Source {i+1}: {source.metadata["source"]}')

Answer: The types of music performed at the orchestra concerts include operatic pieces, orchestral suites, concertos, symphonies, folk songs, and arrangements of classical works. Specific examples include overtures, operatic arias, symphonic compositions by composers like Tchaikovsky, Bruch, and Haydn, as well as instrumental arrangements of folk songs and classical pieces. The concerts feature a variety of styles from different eras, including baroque, classical, romantic, and contemporary music.


Sources:
Source 1: 18. Orchestra Program Spring 2019.pdf
Source 2: Orchestra Program Spring 2009.pdf
Source 3: Indianists _ Dvorak Program.pdf
Source 4: 18. Chamber Orchestra FIRST EDITIONS Spring Program 2017.pdf
Source 5: Spring 5-9-21 Spring Orchestra Program 2021   .pdf
Source 6: Fall 10-10-20 Haverford-Bryn Mawr Orchestra Pop-up Concert Program.pdf
Source 7: 5. Orchestra Program Fall 2014.pdf
Source 8: Spring 4-8-22 Orchestra Program Spring 2022.pdf
Source 9: 4. Orchestra Program Fall 

In [38]:
result = graph.invoke({
    "question": "what are the types of themes performed at the orchestra concerts? How do you define these themes?", "filter" :{"Ensemble_Type": "Orchestra"}
})
#for mroe specificity, asking it to define the themes it is mentioning, also the themes are different than the previous answer. 
print(f'Answer: {result["answer"]}')
print("\n\nSources:")
for i, source in enumerate(result["context"]):
    print(f'Source {i+1}: {source.metadata["source"]}')

Answer: The types of themes performed at the orchestra concerts include harmonic relationships such as the tritone, whole tone scale, and mediant relationships. These themes can be defined as follows:

1. **Tritone**: A musical interval that spans three whole tones, often associated with tension and used in various solos as mentioned in the context (e.g., trombone and trumpet solos).

2. **Whole Tone Scale**: A scale consisting of six tones, each a whole tone apart, which creates a unique, ethereal sound. This is exemplified by the opening theme of “the Sultan.”

3. **Mediant Relationships**: This involves using chords that are a third apart, which create a sense of connection between different keys or tonal areas. This relationship appears frequently in the works of composers like C.P.E. Bach and Beethoven.

These themes contribute to the harmonic progressions and overall structure of the music, providing depth and variety to the orchestral performances.


Sources:
Source 1: 21. Orche

In [39]:
result = graph.invoke({
    "question": "which songs evoke folk themes and cultural identities at the orchestra concert?", "filter" :{"Ensemble_Type": "Orchestra"}
})
#"Evoke" vs "evoked", difference by one letter--> present vs past, and the answers are only a little different. Important to note that when I originally ran this, the answers were completely different. Running it now definitely produced different answers. 
print(f'Answer: {result["answer"]}')
print("\n\nSources:")
for i, source in enumerate(result["context"]):
    print(f'Source {i+1}: {source.metadata["source"]}')

Answer: The songs that evoke folk themes and cultural identities at the orchestra concert are:

1. "Toyos"
2. "Himno de Zampoñas"
3. "Chasqui"
4. "Coqueteos"
5. Folksongs arranged by Luciano Berio:
   - "Black is the colour" (USA)
   - "I wonder as I wander" (USA)
   - "Rossignolet du bois" (France)
   - "A la femminisca" (Sicily)
   - "Ballo" (Italy)
   - "Motettu de Tristura" (Sardinia)
   - "Azerbaijan love song" (Azerbaijan)
6. "Pawnee Horses" by Arthur Farwell
7. "Indian Diary, First Book" by Ferruccio Busoni
8. "Pawnee Preludes" by Curt Cacioppo
9. "American Suite" Op. 98 by Antonín Dvořák

These pieces incorporate elements of various folk traditions and reflect cultural identities.


Sources:
Source 1: Phila Classical Symphony program.pdf
Source 2: Indianists _ Dvorak Program.pdf
Source 3: 8. Orchestra Program Fall 2016.pdf
Source 4: Orchestra Program Spring 2009.pdf
Source 5: Indianists _ Dvorak Program.pdf
Source 6: Indianists _ Dvorak Program.pdf
Source 7: Phila Classical Sym

In [40]:
result = graph.invoke({
    "question": "which songs evoked folk themes and cultural identities at the orchestra concert?", "filter" :{"Ensemble_Type": "Orchestra"}
})
#"Evoke" vs "evoked", difference by one letter--> present vs past, and the answers are only a little different. Important to note that when I originally ran this, the answers were completely different. Running it now definitely produced different answers. 
print(f'Answer: {result["answer"]}')
print("\n\nSources:")
for i, source in enumerate(result["context"]):
    print(f'Source {i+1}: {source.metadata["source"]}')

Answer: The songs that evoked folk themes and cultural identities at the orchestra concert include:

1. Folksongs arranged by Luciano Berio:
   - "Black is the Colour" (USA)
   - "I Wonder as I Wander" (USA)
   - "Rossignolet du bois" (France)
   - "A la femminisca" (Sicily)
   - "Ballo" (Italy)
   - "Motettu de Tristura" (Sardinia)
   - "Azerbaijan Love Song" (Azerbaijan)

2. Leyendas: An Andean Walkabout (2001) by Gabriela Lena Frank, which is based on the idea of mestizaje, blending Western classical and Andean folk music.

3. The thematic material in the Suite commissioned by Polish Radio in 1950, which is based on folk music from the Rzeszów region.

4. Works by Ferruccio Busoni, specifically "Indian Diary, First Book," which includes studies based on Indian songs collected by Natalie Curtis.

5. Pawnee Horses by Arthur Farwell and other Indianist themes represented in the concert.

These pieces represent various cultural identities and folk themes through their use of traditional

## Results

We successfully set up a RAG app with effective filters. We can now "chat" with our data, and "look up" specific information in seconds with a high level of accuracy. The way that this is set up creates a chance of no result, but almost no chance of a purely *incorrect* answer. 

Our filters, however, were very effective in reducing the chance of no answer. Though they currently require a manual call, a future, streamlined version of this project could interpret a question and auto-apply filters with an LLM call. 

Overall, this project resulted in an effective and rapid way to explore our data. 